# CX Intelligence — churn risk pipeline
Jalankan dengan akun kantor. Clear all outputs sebelum menyimpan/share notebook ini.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 2. Ambil kode dari GitHub dan cari folder ml/ secara otomatis
import os, glob, subprocess
REPO = 'https://github.com/athonk2511-del/CX-Intelligence.git'
!rm -rf /content/cx
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, '/content/cx'], capture_output=True, text=True)
print(r.stdout, r.stderr)
hits = glob.glob('/content/cx/**/ml/config.yaml', recursive=True)
if not hits:
    print('GAGAL: ml/config.yaml tidak ditemukan. Isi repo:')
    !find /content/cx -maxdepth 3 -not -path '*/.git*'
    raise SystemExit
ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(ROOT)
print('OK, kode ditemukan di', ROOT)
!pip -q install -r ml/requirements.txt

In [ ]:
# 3. Isi path folder yang berisi Tickets/, Customers/, Work Orders/, CSAT/
DRIVE_ROOT = '/content/drive/MyDrive/CX Intelligence'
import yaml
print('Isi folder:', os.listdir(DRIVE_ROOT))
cfg = yaml.safe_load(open('ml/config.yaml'))
cfg['paths']['drive_root'] = DRIVE_ROOT
yaml.safe_dump(cfg, open('ml/config.local.yaml', 'w'), sort_keys=False, allow_unicode=True)

In [ ]:
# 4. Test
!python -m pytest ml/tests -q

In [ ]:
# 5. Train + score
!python -m ml.churn_pipeline all --config ml/config.local.yaml